# jsteer hello-world: word steering

Fit the model's full Jacobian once (`scripts/fit.py --model ...`, cached to
`artifacts/<model-slug>.jac`), then any word vector is an instant CPU matvec:

```
v_l = unit( J_l^T @ w )
```

`w` is a cotangent (a direction at the output: here the mean unembedding row of
the words you want more or less of). `J_l^T @ w` is the pullback of `w` -- the
standard autodiff name for J-transpose applied to a cotangent -- landing the
concept as a residual-stream direction. This is the verified extraction method
(see the README evidence section). Runtime is steering-lite:
`with v(model, C=...): model.generate(...)`.

In [1]:
# demo notebook authored by Claude
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian

MODEL = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

/media/wassname/SGIronWolf/projects5/2026/jspace/jsteer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 12560.70it/s]

## Fit or load the Jacobian

The expensive step (1 forward + ~d_model/8 backwards per prompt) runs once and
caches to `config.cache_path(MODEL)`. `fit_cached` builds it on first run for
any model and loads it afterwards, so reruns are cheap. Prompts come from jlens's
WikiText corpus. SHOULD: repr shows d_model=1024, source_layers=[8..24].

In [ ]:
# fit-or-load: builds the cache on first run for ANY model, loads it after.
# The lambda means WikiText is only streamed on a cache MISS.
import sys; sys.path.insert(0, "..")  # repo root for config.py
import config
from jlens.examples import load_wikitext_prompts

jac = Jacobian.fit_cached(model, tok, lambda: load_wikitext_prompts(128),
                          config.cache_path(MODEL), layers=(0.3, 0.9))
jac

## Extract a word vector

`word_vector` pulls the words' unembedding direction back through the cached
Jacobian: no gradients, no forward passes, just a matvec per layer.
+C makes the model say these words more, -C less.

In [3]:
v = jac.word_vector(model, tok, ["happy", "joy"])

def gen(vec, prompt, C, do_sample=False, max_new_tokens=40, seed=0):
    enc = tok(prompt, return_tensors="pt").to(model.device)
    torch.manual_seed(seed)
    with vec(model, C=C):
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                             temperature=0.7 if do_sample else None,
                             top_p=0.95 if do_sample else None,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

2026-07-10 13:08:54.515 | INFO     | jsteer.jacobian:_word_cotangent:100 - word cotangent: ['happy', 'joy'] -> first-subtoken ids=[56521, 4123] |w|=0.744


2026-07-10 13:08:54.520 | INFO     | jsteer.jacobian:pullback:189 - jacobian_word per-layer |J^T w| (pre-norm): 8:1.72 9:1.76 10:1.66 11:1.51 12:1.54 13:1.61 14:1.67 15:1.59 16:1.47 17:1.33 18:1.21 19:1.16 20:1.11 21:1.06 22:1.03 23:0.958 24:0.911


## Pick a coefficient: the coherence/strength tradeoff

The raw coefficient is model-dependent. On this 0.6B model a large C
(like 8) overwhelms the residual stream and the output degenerates into
literal "joyjoyjoy..." spam; the interesting regime is small C where the tone
moves but the text stays fluent.

SHOULD: C=0 is neutral; C=1-2 is coherent and noticeably happier; C=4-8
degenerates into token spam. ELSE steering wiring or sign issue.

In [4]:
from tabulate import tabulate

PROMPT = "I went to the park today and"
rows = [(C, gen(v, PROMPT, C)) for C in (0, 1, 2, 4, 8)]
print(tabulate(rows, headers=["C", "greedy generation"], tablefmt="grid", maxcolwidths=[None, 90]))

+-----+--------------------------------------------------------------------------------------------+
|   C | greedy generation                                                                          |
+=====+============================================================================================+
|   0 | I saw a lot of people. I saw a lot of people, and I saw a lot of people again. I saw a     |
|     | lot of people again. I saw a lot of people again. I                                        |
+-----+--------------------------------------------------------------------------------------------+
|   1 | I saw a lot of people there. I was happy with the food and the service. I think it's a     |
|     | good place to visit. I would like to go there again. I think it's                          |
+-----+--------------------------------------------------------------------------------------------+
|   2 | I was happy. I have a good friend, and I love my life. I love the music, the food, 

## Steer at the chosen C

C=1 keeps the model fluent while visibly moving the tone. Greedy and sampled
generations on three different neutral prompts.

In [5]:
C = 1
PROMPTS = [
    "I went to the park today and",
    "The meeting this afternoon was",
    "My overall impression of the new apartment is that",
]
for p in PROMPTS:
    print(f"--- {p!r}")
    print(f"  C=+{C} greedy : {gen(v, p, C)!r}")
    print(f"  C=+{C} sampled: {gen(v, p, C, do_sample=True)!r}")
    print(f"  C=0   greedy : {gen(v, p, 0)!r}")

--- 'I went to the park today and'


  C=+1 greedy : " I saw a lot of people there. I was happy with the food and the service. I think it's a good place to visit. I would like to go there again. I think it's"


  C=+1 sampled: ' I wanted to make some new friends. I saw a cat, and I felt very happy and happy. I wanted to buy a new pair of shoes. I got a new pair of shoes and felt'


  C=0   greedy : ' I saw a lot of people. I saw a lot of people, and I saw a lot of people again. I saw a lot of people again. I saw a lot of people again. I'
--- 'The meeting this afternoon was'


  C=+1 greedy : ' a success. The meeting was a success because the meeting was a success. The meeting was a success because the meeting was a success. The meeting was a success because the meeting was a success. The'


  C=+1 sampled: ' a success. The meeting is very important to me, so I want to share it with you and to let you know that I am very happy. I am happy to have the meeting. What is'


  C=0   greedy : ' held in the library. The meeting was held in the library. The meeting was held in the library. The meeting was held in the library. The meeting was held in the library. The meeting was'
--- 'My overall impression of the new apartment is that'


  C=+1 greedy : " it's a very cozy and warm place. I love the fact that it has a lot of different activities and things to do. I think it's a great place to relax and enjoy the day."


  C=+1 sampled: " it's a beautiful place to live in, but I can't help but be a bit nervous and confused about the details. The first time I came to the apartment, I was really excited and happy"


  C=0   greedy : " it's a very modern and clean apartment. The apartment has a large amount of natural light, which makes the living room feel very open and spacious. The kitchen is also very modern and clean, with"


## Negative steering

The same vector with a negative coefficient suppresses the concept.
SHOULD: less positive affect than C=0, still english (strongly negative C
degenerates the same way strongly positive does).

In [6]:
for p in PROMPTS:
    print(f"--- {p!r}")
    print(f"  C=-2 greedy : {gen(v, p, -2)!r}")

--- 'I went to the park today and'


  C=-2 greedy : ' saw a large amount of wildlife. I noticed that the water surface was covered with sediment from the river. I used a remote sensing system to measure the sediment thickness. What is the sediment thickness in the'
--- 'The meeting this afternoon was'


  C=-2 greedy : ' held in the ______. The ______ was used for the meeting. The ______ was used for the meeting. The ______ was used for the meeting. \n\nFill in the blanks.\n\nThe ______ was used for'
--- 'My overall impression of the new apartment is that'


  C=-2 greedy : ' it is too dark. The windows are too dark. The walls are too dark. The ceiling is too dark. The insulation is too thick. The air quality is too poor. The thermal mass is'


## Bonus: lens readout

Only the full-Jacobian cache gives you this: transport any layer's residual to
the final basis with `J_l` and decode it, i.e. "what does the model think at
layer l". SHOULD: at layer 16 the model has only a city-shaped slot, by layer
20 candidate cities appear, and by layer 24 Paris has won. ELSE layer indexing
is off.

In [7]:
for layer in (16, 20, 24):
    top = jac.lens_topk(model, tok, "The Eiffel Tower is located in the city of", layer=layer, k=6)
    print(f"layer {layer}: {[t for t, _ in top]}")

layer 16: [' town', ' city', ' cities', '_city', ' City', ' Cities']
layer 20: [' Paris', ' London', ' Venice', ' France', ' Berlin', ' Vienna']
layer 24: [' Paris', 'Paris', ' cities', ' Cities', '巴黎', ' City']


## Extract once, steer forever

The steering vector is a plain steering-lite `Vector` (safetensors on disk),
so you can save it and reuse it without the Jacobian cache or jsteer at all.

In [8]:
from steering_lite import Vector

v.save("../artifacts/happy_joy.safetensors")
v2 = Vector.load("../artifacts/happy_joy.safetensors")
print(gen(v2, "I went to the park today and", C=1))

 I saw a lot of people there. I was happy with the food and the service. I think it's a good place to visit. I would like to go there again. I think it's
